# Event Hub Consumer — Wikimedia Recent Changes (Bronze)

Reads events directly from the Event Hub via the
Kafka-compatible endpoint and appends them into Bronze as raw JSON, plus Kafka
delivery metadata (topic/partition/offset) and standard ingestion metadata.

## 1. Configuration

In [0]:
%run "../00-setup/00_config" 

In [0]:
BRONZE_TABLE = f"{BRONZE_SCHEMA}.wikipedia_edits_stream"
CHECKPOINT_PATH = f"{STORAGE_ROOT}/checkpoints/eventhub_to_bronze"

print("Event Hub namespace:", EVENTHUB_NAMESPACE)
print("Event Hub name:", EVENTHUB_NAME)
print("Checkpoint path:", CHECKPOINT_PATH)
print("Target table:", BRONZE_TABLE)

## 2. Load connection string and build Kafka-compatible options

In [0]:
conn_string = dbutils.secrets.get(scope=SECRET_SCOPE_NAME, key=EVENTHUB_SECRET_NAME)
print("Connection string loaded, length:", len(conn_string))

bootstrap_servers = f"{EVENTHUB_NAMESPACE}.servicebus.windows.net:9093"

sasl_config = (
    "kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required "
    'username="$ConnectionString" '
    f'password="{conn_string}";'
)

kafka_options = {
    "kafka.bootstrap.servers": bootstrap_servers,
    "subscribe": EVENTHUB_NAME,
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": sasl_config,
    "startingOffsets": "earliest",
    "failOnDataLoss": "false"
}

print("Bootstrap servers:", bootstrap_servers)

## 3. Read from Event Hub via Kafka source

In [0]:
df_raw_stream = spark.readStream.format("kafka").options(**kafka_options).load()
df_raw_stream.printSchema()

## 4. Decode payload, keep raw JSON + Kafka delivery metadata

In [0]:
from pyspark.sql import functions as F

df_decoded = df_raw_stream.select(
    F.col("value").cast("string").alias("event_json"),
    F.col("topic").alias("kafka_topic"),
    F.col("partition").alias("kafka_partition"),
    F.col("offset").alias("kafka_offset"),
    F.col("timestamp").alias("kafka_enqueued_at"),
)

## 5. Add standard ingestion metadata

In [0]:
df_bronze_stream = (
    df_decoded
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_load_date", F.current_date())
)

## 6. Write to the Bronze Delta table

In [0]:
query = (
    df_bronze_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(BRONZE_TABLE)
)

query.awaitTermination()

row_count = spark.table(BRONZE_TABLE).count()
print("Row count after this run:", row_count)

## 7. Sanity check

In [0]:
display(spark.table(BRONZE_TABLE).orderBy(F.col("_ingested_at").desc()).limit(5))

## 7. Idempotency check — rerun with the same checkpoint

Re-running this notebook without new Event Hub traffic should not add
new rows, since Kafka source offsets are tracked in `CHECKPOINT_PATH`.

In [0]:
before_count = spark.table(BRONZE_TABLE).count()
print(f"Row count BEFORE re-run: {before_count}")

In [0]:
query = (
    df_bronze_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(BRONZE_TABLE)
)
query.awaitTermination()

after_count = spark.table(BRONZE_TABLE).count()
print(f"Row count AFTER re-run: {after_count}")

if after_count == before_count:
    print("✅ IDEMPOTENCY CONFIRMED — no new rows without new Event Hub traffic")
else:
    print(f"ℹ️ Row count changed by {after_count - before_count} — expected if the producer kept sending new events between runs")